In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection

plt.rcParams.update({
    "text.usetex":        False,
    "font.family":        "serif",
    "font.serif":         ["Palatino", "Palatino Linotype", "Georgia",
                           "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset":   "cm",
    "axes.labelsize":     11,
    "axes.titlesize":     11,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "figure.dpi":         150,
})

TROPICAL      = (0.0, 0.46, 0.37)
CTRED         = (177/255, 56/255, 69/255)
CLASSICYELLOW = (191/255, 156/255, 74/255)
LABEL_GREY    = (0.6, 0.6, 0.6)

def _pale(rgb, mix=0.60):
    return tuple(c + (1 - c) * mix for c in rgb)

def _dark(rgb, fac=0.62):
    return tuple(c * fac for c in rgb)

def gradient_path(ax, x, y, centre, cmap, vmax, lw=0.6, alpha=0.85):
    pts  = np.array([x, y], dtype=float).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    vals = (np.abs(y[:-1] - centre) + np.abs(y[1:] - centre)) / 2.0
    norm = mcolors.Normalize(vmin=0, vmax=vmax)
    lc   = LineCollection(segs, cmap=cmap, norm=norm,
                          linewidth=lw, alpha=alpha, rasterized=True)
    lc.set_array(vals)
    ax.add_collection(lc)

def simulate_erw_with_estimator(n, p, q=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    xi = np.empty(n + 1, dtype=np.int8)
    xi[0] = 0
    xi[1] = 1 if rng.random() < q else -1
    for t in range(1, n):
        past = xi[rng.integers(0, t) + 1]
        xi[t + 1] = past if rng.random() < p else -past

    S = np.zeros(n + 1, dtype=np.int64)
    S[1:] = np.cumsum(xi[1:])

    k_range  = np.arange(1, n, dtype=float)
    S_over_k = S[1:n].astype(float) / k_range
    xi_next  = xi[2:n + 1].astype(float)

    num_terms   = S_over_k * (xi_next + S_over_k)
    denom_terms = S_over_k ** 2

    num_cum   = np.cumsum(num_terms)
    denom_cum = np.cumsum(denom_terms)

    with np.errstate(divide='ignore', invalid='ignore'):
        p_hat = num_cum / (2.0 * denom_cum)

    return S, p_hat

P_ROWS   = [(0.60, TROPICAL),
            (0.75, CLASSICYELLOW),
            (0.85, CTRED)]
N_STEPS  = 10_000
Q        = 0.5
BURN_EST = 20
SEED     = 187

rng = np.random.default_rng(SEED)

results = {}
for p_val, _ in P_ROWS:
    S, p_hat = simulate_erw_with_estimator(N_STEPS, p=p_val, q=Q, rng=rng)
    results[p_val] = {"S": S, "p_hat": p_hat}

fig, axes = plt.subplots(3, 2, figsize=(7.2, 6.4))
fig.subplots_adjust(left=0.09, right=0.97, bottom=0.08, top=0.92,
                    wspace=0.38, hspace=0.55)

x_path = np.arange(0, N_STEPS + 1)
n_est  = np.arange(2, N_STEPS + 1)

for row, (p_val, base_col) in enumerate(P_ROWS):
    ax_path = axes[row, 0]
    ax_est  = axes[row, 1]

    cmap = mcolors.LinearSegmentedColormap.from_list(
        f"c{p_val}", [_pale(base_col), base_col])
    dark = _dark(base_col)

    S     = results[p_val]["S"].astype(float)
    p_hat = results[p_val]["p_hat"]

    vmax_path = max(abs(S.min()), abs(S.max())) or 1.0
    gradient_path(ax_path, x_path, S, centre=0.0,
                  cmap=cmap, vmax=vmax_path, lw=0.6, alpha=0.88)

    pad = 0.08 * vmax_path
    ax_path.set_xlim(0, N_STEPS)
    ax_path.set_ylim(S.min() - pad, S.max() + pad)
    ax_path.axhline(0, color="black", lw=0.45, ls=(0, (4, 3)),
                    alpha=0.28, zorder=0)

    mask = n_est >= BURN_EST
    x_e  = n_est[mask].astype(float)
    y_e  = p_hat[mask]

    vmax_est = max(abs(y_e - p_val).max(), 1e-3)
    gradient_path(ax_est, x_e, y_e, centre=p_val,
                  cmap=cmap, vmax=vmax_est, lw=0.75, alpha=0.95)

    ymin = min(y_e.min(), p_val)
    ymax = max(y_e.max(), p_val)
    pad_y = max(0.04, 0.18 * (ymax - ymin))
    ax_est.set_xlim(0, N_STEPS)
    ax_est.set_ylim(ymin - pad_y, ymax + pad_y)

    ax_est.axhline(p_val, color="black", lw=0.55, ls=(0, (5, 3)),
                   alpha=0.55, zorder=4)

    for ax in (ax_path, ax_est):
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(0.55)
        ax.spines["bottom"].set_linewidth(0.55)
        ax.tick_params(width=0.55, length=3, direction="out")
        ax.xaxis.set_major_locator(
            ticker.FixedLocator([0, N_STEPS // 2, N_STEPS]))
        ax.xaxis.set_major_formatter(
            ticker.FuncFormatter(lambda x, _: f"${int(x)}$"))

    ax_path.yaxis.set_major_locator(ticker.MaxNLocator(nbins=3, symmetric=True))
    ax_est.yaxis.set_major_locator(ticker.MaxNLocator(nbins=3))

    ax_path.set_ylabel(r"$S_n$", labelpad=4)
    ax_est.set_ylabel(r"$\widehat{p}_n$", labelpad=4)

    if row == len(P_ROWS) - 1:
        ax_path.set_xlabel(r"$n$", labelpad=3)
        ax_est.set_xlabel(r"$n$", labelpad=3)

    box_left  = ax_path.get_position()
    box_right = ax_est.get_position()
    x_centre  = 0.5 * (box_left.x0 + box_right.x1)
    y_top     = box_left.y1 + 0.025
    fig.text(x_centre, y_top,
             rf"$p = {p_val:.2f}$",
             ha="center", va="bottom", fontsize=9, color=LABEL_GREY)

top_left  = axes[0, 0].get_position()
top_right = axes[0, 1].get_position()
fig.text(0.5 * (top_left.x0 + top_left.x1),
         top_left.y1 + 0.06,
         r"sample path",
         ha="center", va="bottom", fontsize=9.5,
         color=LABEL_GREY, style="italic")
fig.text(0.5 * (top_right.x0 + top_right.x1),
         top_right.y1 + 0.06,
         r"running estimator $\widehat{p}_n$",
         ha="center", va="bottom", fontsize=9.5,
         color=LABEL_GREY, style="italic")

fig.savefig("/home/claude/fig_estimator.pdf", bbox_inches="tight")
fig.savefig("/home/claude/fig_estimator.png", bbox_inches="tight", dpi=220)
print("Saved fig_estimator.pdf and fig_estimator.png")